# CSP Lab Demo — Australia Map Coloring

This notebook uses the **Australia map-coloring problem** from the supplied CSP lecture.

We will learn:
- Variables, domains and constraints
- Constraint graphs
- Backtracking
- MRV (Minimum Remaining Values)
- LCV (Least Constraining Value)
- Forward checking
- AC-3 (Arc Consistency)

A CSP represents a problem using variables with domains and constraints on their values.
The lecture's Australia example uses WA, NT, Q, NSW, V, SA and T, with three colors and
the constraint that adjacent regions must have different colors.

## 1. CSP = Variables + Domains + Constraints

### Australia map-coloring

**Variables**

`WA, NT, Q, NSW, V, SA, T`

**Domain**

`{Red, Green, Blue}`

**Constraint**

Adjacent regions must have different colors.

For example:

`WA != NT`

`WA != SA`

`NT != SA`

A **solution** is a complete assignment in which every constraint is satisfied.

In [ ]:
variables = ["WA", "NT", "SA", "Q", "NSW", "V", "T"]
colors = ["Red", "Green", "Blue"]

domains = {v: colors.copy() for v in variables}

neighbors = {
    "WA": {"NT", "SA"},
    "NT": {"WA", "SA", "Q"},
    "SA": {"WA", "NT", "Q", "NSW", "V"},
    "Q": {"NT", "SA", "NSW"},
    "NSW": {"SA", "Q", "V"},
    "V": {"SA", "NSW"},
    "T": set()
}

print("Variables:", variables)
print("Domain:", colors)
print("Neighbors of SA:", neighbors["SA"])

## 2. Constraint graph

A constraint graph represents:

- **Node = variable**
- **Edge = constraint**

For Australia map coloring, an edge means the two regions must have different colors.

Tasmania (`T`) has no edge to the mainland in this simplified CSP, so it is an independent subproblem.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

G = nx.Graph()
G.add_nodes_from(variables)

for v in variables:
    for n in neighbors[v]:
        G.add_edge(v, n)

pos = {
    "WA": (0, 1), "NT": (1, 2), "SA": (1, 0),
    "Q": (2, 2), "NSW": (2, 0.7), "V": (2.5, -0.2),
    "T": (4, -0.5)
}

nx.draw(G, pos, with_labels=True, node_size=1800, font_size=12)
plt.title("Australia CSP Constraint Graph")
plt.show()

## 3. Backtracking

Backtracking assigns **one variable at a time**.

At each step:

1. Choose an unassigned variable.
2. Try a value.
3. Check the constraints.
4. If valid, continue recursively.
5. If it leads to failure, undo the assignment and try another value.

Think:

**choose → assign → check → continue → failure → backtrack**

In [ ]:
steps = 0

def is_consistent(var, value, assignment):
    for n in neighbors[var]:
        if n in assignment and assignment[n] == value:
            return False
    return True

def backtracking(assignment):
    global steps
    steps += 1

    if len(assignment) == len(variables):
        return assignment.copy()

    var = next(v for v in variables if v not in assignment)

    for value in colors:
        if is_consistent(var, value, assignment):
            assignment[var] = value

            result = backtracking(assignment)
            if result is not None:
                return result

            del assignment[var]

    return None

steps = 0
solution = backtracking({})
print("Solution:", solution)
print("Search steps:", steps)

## 4. MRV — Minimum Remaining Values

MRV asks:

> **Which unassigned variable has the fewest legal values right now?**

We choose that variable first.

This tries to discover difficult choices and failures early.

In [ ]:
def legal_values(var, assignment, current_domains=None):
    d = current_domains[var] if current_domains else colors
    return [value for value in d if is_consistent(var, value, assignment)]

def select_mrv(assignment):
    unassigned = [v for v in variables if v not in assignment]
    return min(unassigned, key=lambda v: len(legal_values(v, assignment)))

def lcv_order(var, assignment, current_domains=None):
    d = current_domains[var] if current_domains else colors
    values = legal_values(var, assignment, current_domains)

    def ruled_out(value):
        count = 0
        for n in neighbors[var]:
            if n not in assignment:
                nd = current_domains[n] if current_domains else colors
                if value in nd:
                    count += 1
        return count

    return sorted(values, key=ruled_out)

def mrv_lcv_backtracking(assignment):
    if len(assignment) == len(variables):
        return assignment.copy()

    var = select_mrv(assignment)
    print("MRV selected:", var)

    for value in lcv_order(var, assignment):
        print("  LCV tries:", value)
        assignment[var] = value

        result = mrv_lcv_backtracking(assignment)
        if result is not None:
            return result

        del assignment[var]

    return None

solution = mrv_lcv_backtracking({})
print("\nSolution:", solution)

## 5. Forward checking

After assigning a value, remove that value from the domains of unassigned neighbors.

If any future variable gets an empty domain, stop immediately and backtrack.

So forward checking asks:

> **Did my current assignment make some future variable impossible?**

In [ ]:
def forward_check(var, value, assignment, current_domains):
    for n in neighbors[var]:
        if n not in assignment:
            if value in current_domains[n]:
                current_domains[n].remove(value)

            if not current_domains[n]:
                return False

    return True

def fc_search(assignment, current_domains):
    if len(assignment) == len(variables):
        return assignment.copy()

    unassigned = [v for v in variables if v not in assignment]
    var = min(unassigned, key=lambda v: len(current_domains[v]))

    for value in current_domains[var].copy():
        if not is_consistent(var, value, assignment):
            continue

        new_assignment = assignment.copy()
        new_assignment[var] = value

        new_domains = {v: current_domains[v].copy() for v in variables}

        if forward_check(var, value, new_assignment, new_domains):
            result = fc_search(new_assignment, new_domains)
            if result is not None:
                return result

    return None

solution = fc_search({}, {v: colors.copy() for v in variables})
print("Forward-checking solution:", solution)

## 6. AC-3 — Arc Consistency

For an arc `X → Y`, every value in X must have at least one compatible value in Y.

For our constraint `X != Y`, a value `x` can remain in X only if Y has some value different from x.

AC-3:
1. puts all arcs in a queue;
2. checks an arc;
3. removes unsupported values;
4. if a domain changes, rechecks neighboring arcs;
5. reports failure if a domain becomes empty.

In [ ]:
from collections import deque

def revise(X, Y, current_domains):
    revised = False

    for x in current_domains[X].copy():
        if not any(x != y for y in current_domains[Y]):
            current_domains[X].remove(x)
            revised = True

    return revised

def ac3(current_domains):
    queue = deque(
        (X, Y)
        for X in variables
        for Y in neighbors[X]
    )

    while queue:
        X, Y = queue.popleft()

        if revise(X, Y, current_domains):
            if not current_domains[X]:
                return False

            for Z in neighbors[X]:
                if Z != Y:
                    queue.append((Z, X))

    return True

test_domains = {v: colors.copy() for v in variables}

print("Before AC-3:")
print(test_domains)

ok = ac3(test_domains)

print("\nAC-3 successful:", ok)
print("After AC-3:")
print(test_domains)

## 7. AC-3 + Backtracking

AC-3 can be used after making an assignment.

```text
Choose variable
      ↓
Choose value
      ↓
Assign value
      ↓
Run AC-3
      ↓
Any empty domain?
   ↙       ↘
 YES        NO
  ↓          ↓
Backtrack   Continue
```

This allows constraint propagation to detect some failures before the search goes much deeper.

In [ ]:
def ac3_search(assignment, current_domains):
    if len(assignment) == len(variables):
        return assignment.copy()

    unassigned = [v for v in variables if v not in assignment]
    var = min(unassigned, key=lambda v: len(current_domains[v]))

    for value in lcv_order(var, assignment, current_domains):
        if not is_consistent(var, value, assignment):
            continue

        new_assignment = assignment.copy()
        new_assignment[var] = value

        new_domains = {v: current_domains[v].copy() for v in variables}
        new_domains[var] = [value]

        if ac3(new_domains):
            result = ac3_search(new_assignment, new_domains)
            if result is not None:
                return result

    return None

solution = ac3_search({}, {v: colors.copy() for v in variables})
print("AC-3 enhanced solution:", solution)

## 8. Summary

| Method | Main idea |
|---|---|
| Backtracking | Assign one variable at a time and undo on failure |
| MRV | Choose the variable with the fewest legal values |
| LCV | Try the value that constrains neighbors the least |
| Forward Checking | Remove values that immediately conflict with an assignment |
| AC-3 | Repeatedly propagate constraints between variables |

### Quick memory trick

**MRV → Which variable?**

**LCV → Which value?**

**Forward checking → What did my assignment immediately make impossible?**

**AC-3 → What else can constraint propagation make impossible?**

## 9. Questions for discussion

1. What are the variables in this CSP?
2. What is the domain?
3. What does an edge in the constraint graph represent?
4. Why is Tasmania independent in this simplified problem?
5. What happens when backtracking reaches a dead end?
6. How does MRV select a variable?
7. How does LCV select a value?
8. What is the difference between forward checking and AC-3?
9. Why can AC-3 detect some failures earlier?